<a href="https://colab.research.google.com/github/roberthsu2003/machine_learning/blob/main/%E5%A4%9A%E5%85%83%E7%B7%9A%E6%80%A7%E8%BF%B4%E6%AD%B8/multiple_linear_regression_insurance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 多元線性迴歸實戰：醫療保險費用預測 (Medical Insurance Costs Prediction)

在這個實作中，我們將使用真實世界的醫療保險費用數據集 `insurance.csv`，透過多元線性迴歸模型來預測保險收費。這個範例非常適合用來學習如何同時處理「數值特徵」與「類別特徵（文字）」並存的真實世界機器學習問題。

---

### 💡 數據背景與生活對照（導讀）

#### 1. 什麼是「保險費用」？為什麼每個人交的保費不固定？
在商業醫療保險中，保費是**因人而異（不固定）**的，這在保險學中稱為**「風險差異定價」**：
- **核心邏輯**：健康風險越高的人，未來生病住院的機率越高，保險公司收取的保費就越貴。
- **變數（特徵）**：包含保戶的**年齡（Age）**、**BMI（身體質量指數）**、**是否抽菸（Smoker）**與**居住地（Region）**等。
- **目標值（應變數）**：**年保費金額（Charges，單位為美元）**。

#### 2. 這代表哪裡的保險制度？與台灣的健保有何不同？
- **美國商業保險制度**：此數據集反映的是美國高度私有化的商業醫療保險體系。美國沒有像台灣一樣的全民強制社會健保，大多數人需向私營保險公司購買保費高低不等且與個人風險掛鉤的保險。
- **與台灣健保的對照**：
  - **台灣全民健保（社會保險）**：保費多寡主要取決於個人的**薪資收入高低**，與您是否抽菸、BMI 多少無關。
  - **台灣商業保險（自費險）**：如果您去購買國泰、富邦等保險公司的商業醫療險（例如防癌險、實支實付），保險公司同樣會採用本範例中的邏輯，依照**年齡、性別、甚至是否抽菸**來進行差別定價。


In [ ]:
# 匯入必要的套件
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn import metrics

# 載入保險數據集
data = pd.read_csv('insurance.csv')
print("資料集維度:", data.shape)
data.head()

### 1. 數據預處理 (Data Preprocessing)

此數據集包含文字欄位（類別特徵）與數字欄位（數值特徵）：
- **二元類別特徵**：`sex`（female/male）與 `smoker`（yes/no），我們可以使用 Label Mapping 直接映射轉換。
- **多元類別特徵**：`region`（4個地區：southwest, southeast, northwest, northeast），我們使用 One-Hot Encoding 進行轉碼。
- **數值特徵**：`age`、`bmi`、`children`，稍後會使用標準化進行特徵縮放。

In [ ]:
# 對二元類別欄位進行映射 (Label Encoding)
data['sex'] = data['sex'].map({'female': 0, 'male': 1})
data['smoker'] = data['smoker'].map({'no': 0, 'yes': 1})
data.head()

In [ ]:
# 使用 pandas 的 get_dummies 進行 One-Hot Encoding
# drop_first=True 可以避免「虛擬變數陷阱 (Dummy Variable Trap)」，從而防止特徵間產生完全多重共線性。
# 這會產生 region_northwest, region_southeast, region_southwest 三個欄位 (以 region_northeast 作為基準欄位)
data = pd.get_dummies(data, columns=['region'], drop_first=True)

# 如果產生出來的欄位是布林值 (True/False)，我們將其轉換為整數 (0/1)
bool_cols = [col for col in data.columns if data[col].dtype == 'bool']
if bool_cols:
    data[bool_cols] = data[bool_cols].astype(int)

data.head()

### 2. 劃分特徵與目標變數，並分割訓練集與測試集

我們將 `charges` 作為目標變數 $y$（保險費用），其餘欄位作為特徵 $X$。並以 80% 訓練集、20% 測試集的比例進行隨機切分。

In [ ]:
X = data.drop('charges', axis=1)
y = data['charges']

# 切分訓練集與測試集
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"訓練集樣本數: {X_train.shape[0]}, 測試集樣本數: {X_test.shape[0]}")

### 3. 特徵縮放 (Feature Scaling)

為了解決各特徵尺度懸殊的問題，我們使用 `StandardScaler` 將數值特徵（`age`、`bmi`、`children`）進行標準化（均值為 0，標準差為 1）。

> **注意**：我們僅對數值型特徵進行縮放，已經轉換為 0/1 的二元編碼與獨熱編碼特徵則保持原樣即可。

In [ ]:
# 定義需要標準化的數值特徵欄位
num_features = ['age', 'bmi', 'children']

# 建立標準化器
scaler = StandardScaler()

# 複製資料集以免修改到原始變數
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

# 僅在訓練集上 fit，並對訓練集與測試集進行 transform (防止 Data Leakage)
X_train_scaled[num_features] = scaler.fit_transform(X_train[num_features])
X_test_scaled[num_features] = scaler.transform(X_test[num_features])

X_train_scaled.head()

### 4. 建立與訓練多元線性迴歸模型

使用 Scikit-Learn 的 `LinearRegression` 類別在標準化後的數據上進行擬合訓練。

In [ ]:
# 初始化線性迴歸模型
model = LinearRegression()

# 訓練模型
model.fit(X_train_scaled, y_train)

# 對測試集進行預測
y_pred = model.predict(X_test_scaled)

### 5. 模型評估 (Model Evaluation)

我們使用平均絕對誤差 (MAE)、均方誤差 (MSE)、均方根誤差 (RMSE) 與決定係數 ($R^2$ Score) 來客觀評估模型效能。

In [ ]:
mae = metrics.mean_absolute_error(y_test, y_pred)
mse = metrics.mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = metrics.r2_score(y_test, y_pred)

print(f"平均絕對誤差 (MAE): {mae:.2f} USD")
print(f"均方誤差 (MSE): {mse:.2f}")
print(f"均方根誤差 (RMSE): {rmse:.2f} USD")
print(f"決定係數 (R² Score): {r2:.4f}")

### 6. 模型係數解讀 (Coefficient Interpretation)

多元線性迴歸具備強大的「可解釋性」。我們可以將每個特徵所對應的權重（係數 Coefficient）整理成表格，探討它們如何影響最終的醫療保險費用。

In [ ]:
# 輸出截距與係數表
print(f"模型截距 (Intercept b): {model.intercept_:.2f} USD\n")

coeff_df = pd.DataFrame(model.coef_, X.columns, columns=['Coefficient'])
coeff_df

### 💡 模型預測結果與係數解讀探討

透過以上的模型係數，我們能得出非常有趣的結論，並向學生說明這些係數的物理/實務含意：

1. **是否抽菸 (`smoker`)**：係數高達 **23,651 USD**！這意味著在其他條件（年齡、BMI、居住地、孩子數量）完全相同的情況下，**抽菸者比不抽菸者的保險費用平均高出 23,651 美元**。抽菸是影響醫療費用最關鍵的因子。
2. **年齡 (`age`)**：年齡每增加 1 個標準差，保費平均增加約 **3,613 USD**。這符合常理，年齡越大，醫療風險越高。
3. **身體質量指數 (`bmi`)**：BMI 每增加 1 個標準差，保費平均增加約 **2,036 USD**。過重會帶來更高的健康風險。
4. **性別 (`sex`)**：男性的係數僅為 **-18.60 USD**（相當微弱且接近 0），這說明性別在控制了其他因素後，對保費的影響幾乎可以忽略。
5. **地區 (`region`)**：相較於基準地區 northeast，其他地區（northwest, southeast, southwest）的係數均為負值，且數值在 -300 至 -900 左右，顯示不同地區的收費有微小的差異，但遠不如抽菸習慣與年齡的影響顯著。

### 7. 對新數據進行保費預測 (Predicting on New Data)

現在，我們建立 3 筆虛擬的保戶新數據，並使用我們訓練好的多元線性迴歸模型來預測他們每年需要繳交的保險費用。

- **保戶 A**：19 歲，不抽菸，BMI 為 22.0（標準體重），沒有小孩，住在西北區 (northwest)。
- **保戶 B**：45 歲，抽菸，BMI 為 30.0（中度肥胖），有 2 個小孩，住在東南區 (southeast)。
- **保戶 C**：60 歲，不抽菸，BMI 為 25.0（微胖過重），有 1 個小孩，住在東北區 (northeast)。

> **💡 核心觀念提醒**：
> 在將新資料送入模型預測前，新資料必須進行**與訓練資料完全相同的前處理步驟**（包含二元映射與獨熱編碼對齊）。特別是針對數值特徵的標準化，必須使用**訓練時擬合的同一個 `scaler`** 進行 `.transform()`！

In [ ]:
# 創建新數據的 DataFrame (特徵順序與欄位名稱必須與特徵矩陣 X 完全一致)
new_data = pd.DataFrame([
    # 保戶 A: 19歲, 女性(0), BMI 22.0, 0個小孩, 不抽菸(0), northwest(1), southeast(0), southwest(0)
    [19, 0, 22.0, 0, 0, 1, 0, 0],
    # 保戶 B: 45歲, 男性(1), BMI 30.0, 2個小孩, 抽菸(1), northwest(0), southeast(1), southwest(0)
    [45, 1, 30.0, 2, 1, 0, 1, 0],
    # 保戶 C: 60歲, 女性(0), BMI 25.0, 1個小孩, 不抽菸(0), northwest(0), southeast(0), southwest(0) (即東北地區基準)
    [60, 0, 25.0, 1, 0, 0, 0, 0]
], columns=X.columns)

# 複製新數據集進行縮放
new_data_scaled = new_data.copy()

# 對新數據中的數值特徵進行標準化轉換 (使用前面 fit 好的 scaler)
new_data_scaled[num_features] = scaler.transform(new_data[num_features])

# 使用訓練好的模型進行預測
predictions = model.predict(new_data_scaled)

# 將預測結果寫回原始數據表，以便人類閱讀
new_data_display = new_data.copy()
new_data_display['sex'] = new_data_display['sex'].map({0: 'female', 1: 'male'})
new_data_display['smoker'] = new_data_display['smoker'].map({0: 'no', 1: 'yes'})

# 還原地區特徵以簡化表格
def get_region(row):
    if row['region_northwest'] == 1: return 'northwest'
    if row['region_southeast'] == 1: return 'southeast'
    if row['region_southwest'] == 1: return 'southwest'
    return 'northeast'

new_data_display['region'] = new_data_display.apply(get_region, axis=1)
# 刪除獨熱編碼欄位
new_data_display = new_data_display.drop(['region_northwest', 'region_southeast', 'region_southwest'], axis=1)

# 加入預測結果保費欄位
new_data_display['Predicted_Charges_USD'] = np.round(predictions, 2)
new_data_display